In [3]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=1000, distance_threshold = 0.2, use_mst=True, enrich=False):

    data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, label_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    print("Done indexing")

    data_hetero, labels = gnn_utils.build_hetero_graph(
                            data,
                            chephy_matrix,
                            sequences_list,
                            epitopes,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    return data_hetero, data


In [14]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
input_file = "/home/dsi/chenbis/repos/sol_lab/data/train.tsv"
data = run.load_dataframe(input_file)
label_header = "peptide"
cdr3_header = "cdr3"

n = len(data)
distance_threshold=0.05
use_mst=False
enrich=True

data_hetero_train, data_train = create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

In [ ]:
input_file = "/home/dsi/chenbis/repos/sol_lab/data/test.tsv"
data = run.load_dataframe(input_file)
label_header = "peptide"
cdr3_header = "cdr3"
n = len(data)
distance_threshold=0.05
use_mst=False
enrich=True

data_hetero_test, data_test = create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

In [ ]:
model = GNN_model.HeteroTCR(data_hetero_train.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_train = data_hetero_train.to(device)
data_hetero_test = data_hetero_test.to(device)

In [ ]:
# Create edge label indices for training data
train_edge_pairs = data_train[['tcr_index', 'epitope_index']].values
train_edge_label_index = torch.tensor(train_edge_pairs.T)

# Create edge label indices for test data
test_edge_pairs = data_test[['tcr_index', 'epitope_index']].values
test_edge_label_index = torch.tensor(test_edge_pairs.T)

# Create labels
y_train = data_train['Binding'].values
y_test = data_test['Binding'].values

In [ ]:
import os
import torch
import pandas as pd


save_model_path = f'/home/dsi/chenbis/repos/sol_lab/output/models/{run.get_time()}'
os.makedirs(save_model_path, exist_ok=True)
NUM_EPOCHS = 1000

# Track performance
epoch = []
loss = []
acc = []
auc_roc = []
val_loss = []
val_acc = []
val_auc_roc = []

# Best model tracking
best_val_auc = -float('inf')
best_val_loss = float('inf')
best_auc_model_state = None
best_loss_model_state = None
best_auc_epoch = 0
best_loss_epoch = 0

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0)

for ep in range(1, NUM_EPOCHS + 1):
    train_loss, train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC = GNN_model.train(
        model, optimizer,
        data_hetero_train, data_hetero_test,
        train_edge_label_index, test_edge_label_index,
        y_train, y_test,
        device
    )

    print(
        'Train epoch: {} - loss: {:.4f} - binary_accuracy: {:.4f} - ROCAUC: {:.4f} - val_loss: {:.4f} - val_binary_accuracy: {:.4f} - val_ROCAUC: {:.4f}'.format(
            ep, train_loss.item(), train_binary_accuracy, train_ROCAUC, valid_loss, val_binary_accuracy, val_ROCAUC))

    # Save best AUC model
    if val_ROCAUC > best_val_auc:
        best_val_auc = val_ROCAUC
        best_auc_model_state = model.state_dict()
        best_auc_epoch = ep

    # Save best loss model
    if valid_loss < best_val_loss:
        best_val_loss = valid_loss
        best_loss_model_state = model.state_dict()
        best_loss_epoch = ep

    # Store metrics
    epoch.append(ep)
    loss.append(train_loss.item())
    acc.append(train_binary_accuracy.item())
    auc_roc.append(train_ROCAUC.item())
    val_loss.append(valid_loss.item())
    val_acc.append(val_binary_accuracy.item())
    val_auc_roc.append(val_ROCAUC.item())

# Save training history
root_history = '/home/dsi/chenbis/repos/sol_lab/output'
dfhistory = {
    'epoch': epoch,
    'loss': loss, 'acc': acc, 'auc_roc': auc_roc,
    'val_loss': val_loss, 'val_acc': val_acc, 'val_auc_roc': val_auc_roc
}
df = pd.DataFrame(dfhistory)
df.to_csv(os.path.join(root_history, "history.csv"), header=True, index=False)

# Save best models to disk
torch.save(best_auc_model_state, os.path.join(save_model_path, f"max_AUC_HeteroTCR_epoch{best_auc_epoch:03d}_AUC{best_val_auc:.6f}.pth"))
torch.save(best_loss_model_state, os.path.join(save_model_path, f"minloss_AUC_HeteroTCR_epoch{best_loss_epoch:03d}_LOSS{best_val_loss:.6f}.pth"))

print("Best AUC model saved from epoch:", best_auc_epoch, "with AUC:", best_val_auc)
print("Best loss model saved from epoch:", best_loss_epoch, "with loss:", best_val_loss)


In [1]:
def generate_negative_edges(df, num_negatives, negative_score=0.0):
    """
    Efficiently generate negative cdr3~epitope pairs not present in df.
    """
    cdr3_pool = df['cdr3'].unique()
    epitope_pool = df['epitope'].unique()

    # Create full cartesian product
    all_combinations = pd.DataFrame(
        [(cdr3, epitope) for cdr3 in cdr3_pool for epitope in epitope_pool],
        columns=['cdr3', 'epitope']
    )

    # Remove positive (existing) pairs
    positives = df[['cdr3', 'epitope']].drop_duplicates()
    merged = all_combinations.merge(positives, on=['cdr3', 'epitope'], how='left', indicator=True)
    negatives = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

    # Shuffle and sample
    negatives = negatives.sample(n=min(num_negatives, len(negatives)), random_state=42).copy()
    negatives['score'] = negative_score
    negatives['source'] = 'NEGATIVE'

    # Truncate cdr3
    def truncate(seq):
        mid = (len(seq) + 1) // 2
        return seq[max(0, mid - 4):min(len(seq), mid + 4)]
    negatives['cdr3_truncated'] = negatives['cdr3'].apply(truncate)

    return negatives



def filter_data(df, source=None, min_score=None, max_score=None, num_negatives=0, negative_score=0.0):
    """
    Filters the combined DataFrame by source and score range, with optional negative edge generation.

    Parameters:
        df (pd.DataFrame): Combined dataset with 'source' and 'score' columns.
        source (str or list of str, optional): Filter by source(s) (e.g., "VDJDB" or ["McPAS", "TRAIT"]).
        min_score (float, optional): Minimum score (inclusive).
        max_score (float, optional): Maximum score (inclusive).
        num_negatives (int, optional): Number of negative cdr3~epitope pairs to generate.
        negative_score (float, optional): Score assigned to negative edges (default is 0.0).

    Returns:
        pd.DataFrame: Filtered DataFrame with optional negative edges appended.
    """
    filtered_df = df.copy()

    # Filter by source
    if source is not None:
        if isinstance(source, str):
            source = [source]
        filtered_df = filtered_df[filtered_df['source'].isin(source)]

    # Filter by score range
    if min_score is not None:
        filtered_df = filtered_df[filtered_df['score'] >= min_score]
    if max_score is not None:
        filtered_df = filtered_df[filtered_df['score'] <= max_score]

    # Efficient negative generation
    if num_negatives > 0:
        negative_df = generate_negative_edges(filtered_df, num_negatives, negative_score)
        filtered_df = pd.concat([filtered_df, negative_df], ignore_index=True)
    return filtered_df


In [10]:
data_combined = pd.read_csv('/home/dsi/chenbis/repos/sol_lab/files/combined_data.csv')
data_combined = filter_data(data_combined, source=["VDJDB"], num_negatives=10000, min_score=2, max_score=3)

cdr3_header = "cdr3"
label_header = "epitope"

n = len(data_combined)
distance_threshold=0.05
use_mst=False
enrich=False

combined_hetero, data_combined = create_hetero_data(data_combined, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)


sequence_vectors done on GPU
pdist done on GPU
normalized done on GPU
moved to CPU
Done indexing
skip mst
Step 1, Done!
Step 2, Done!
Step 3, Done!
Step 4, Done!
Step 5, Done!


In [11]:

# Create edge label indices for test data
combined_edge_pairs = data_combined[['tcr_index', 'epitope_index']].values
combined_edge_label_index = torch.tensor(combined_edge_pairs.T)

# Create labels
y_combined = np.where(data_combined['source'] == 'NEGATIVE', 0, 1)

In [ ]:
model = GNN_model.HeteroTCR(combined_hetero.metadata(), 1024, 3, "SAGE")
model = model.to(device)
combined_hetero = combined_hetero.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(combined_hetero.x_dict, combined_hetero.edge_index_dict, combined_edge_label_index)

model_dir = save_model_path
val_model_path = save_model_path
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, combined_hetero, combined_edge_label_index, y_combined)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

In [ ]:
# Create a new dataframe with tcr, epitope, true label, and predicted binding probability
results_df = pd.DataFrame({
    'tcr': data_combined['cdr3'],
    'epitope': data_combined['epitope'],
    'true_label': y_combined,
    'predicted_binding_prob': test_prob.cpu().numpy()
})

print(results_df.head())

In [ ]:
model = GNN_model.HeteroTCR(data_hetero_test.metadata(), 1024, 3, "SAGE")
model = model.to(device)
data_hetero_test = data_hetero_test.to(device)

with torch.no_grad():  # Initialize lazy modules.
    out = model(data_hetero_test.x_dict, data_hetero_test.edge_index_dict, test_edge_label_index)

model_dir = save_model_path
val_model_path = save_model_path
for root, dirs, files in os.walk(val_model_path):
    for file in files:
        if file.startswith("min"):
            PATH = os.path.join(val_model_path, file)
            model.load_state_dict(torch.load(PATH))
            test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, data_hetero_test, test_edge_label_index, y_test)
            print("ACC: {:.4f}".format(test_binary_accuracy))
            print("AUC: {:.4f}".format(test_ROCAUC))

In [ ]:
def load_your_trained_model(model_path, metadata):
    model = GNN_model.HeteroTCR(metadata=metadata)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

def prediction_func(df_input, model_path=None, **kwargs):
    df_output = df_input.copy()

    save_model_path = "/home/dsi/chenbis/repos/sol_lab/output/models20250609_191843/minloss_AUC_HeteroTCR_epoch294_LOSS0.631632.pth"
    cdr3_header = "CDR3_beta"
    label_header = "Epitope"

    n = len(df_input)
    distance_threshold=0.05
    use_mst=False
    enrich=False

    hetero, data = create_hetero_data(df_input, label_header, cdr3_header, atchley_dict, n=n, distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

    # Create edge label indices for test data
    edge_pairs = data[['tcr_index', 'epitope_index']].values
    edge_label_index = torch.tensor(edge_pairs.T)

    # Load trained model
    model = load_your_trained_model(save_model_path, hetero.metadata())
    model = model.to(device)
    # Prepare edge_label_index
    edge_label_index = hetero['tcr', 'binds', 'epitope'].edge_index

    # Run prediction
    with torch.no_grad():
        out = model(hetero.x_dict, hetero.edge_index_dict, edge_label_index)
        scores = out.cpu().numpy()

    df_output["Score"] = scores
    # Add predictions to output DataFrame
    return df_output


In [34]:
import numpy as np

def prediction_func(df_input, **kwargs):
    """
    This function needs to serve as a wrapper function between your model and the test. 
    :param df_input: pandas.DataFrame with columns:
        - CDR3_alpha, CDR3_beta: CDR3 amino acid region of the alpha and beta chains starting with 'C' and ending with 'F' or 'W' (e.g. 'CASSMRSAVEQYF')
        - V_alpha, V_beta, J_alpha, J_beta: Categorical V,J-gene annotation (e.g. TRBJ2-7*01) 
        - Epitope: amino acid sequence of the epitope (e.g. 'GILGFVFTL')
        - MHC: Categorical annotation of the MHC type 
        - clone_id: identifier ID of this clonotype without any predictive meaning
        - dataset: name of the dataset without any predictive meaning
        - Label: 1 for binder, 0 for none-binder, DO NOT USE THIS LABEL IN ANY PART OF YOUR MODEL
       Note: while the DataFrame provides all these inputs, feel free to use only parts of them for your model 
    :param kwargs: Any additional parameters (e.g. different model choices) can be passed as keyword arguments here at your choosing, which can be passed through the config (see below)

    :return: pandas.DataFrame of columns:
        - the same as df_input
        - Score: numeric prediction score, where higher values indicate a higher probability of binding for this row (TCR-Epitope pair)
    """
    # Let's print the input to the function to get an example later
    print(df_input.columns)
    print(df_input.head(5))
    print(kwargs)

    # The output should follow the same format as the input but additionally contains the prediction score
    df_output = df_input.copy()

    # Here you need to plug in your model including preprocessing (e.g. reconstruct full TCR sequence, prune leading 'C' from CDR3, rename the columns, ...)
    # For the dummy model, we simply assign a random score without any relationship to the input
    np.random.seed(0)  # Seed your model for reproducability
    df_output["Score"] = np.random.rand(len(df_output))

    return df_output

In [35]:
from benchmarks.tcr_benchmark.eval.allTests import NAME_2_TEST
from benchmarks.tcr_benchmark.pp.datasets import download_datasets

predictor_name = 'hetero_gnn'
dataset = 'viral'
download_datasets(dataset)
test = NAME_2_TEST[dataset](None)

config = {'model_path': '/home/dsi/chenbis/repos/sol_lab/output/models20250609_191843/minloss_AUC_HeteroTCR_epoch294_LOSS0.631632.pth'}  # point to your trained weights
# Ensure all tensors are on the same device

config = {'dummyModelChoice': 'Use the super model'}

results = test.run_tests(prediction_func, predictor_name, config)
results.head()


100%|██████████| 3/3 [00:00<00:00, 938.25it/s]


Index(['CDR3_alpha', 'CDR3_beta', 'V_alpha', 'V_beta', 'J_alpha', 'J_beta',
       'Epitope', 'clone_id', 'MHC', 'dataset', 'Label'],
      dtype='object')
          CDR3_alpha       CDR3_beta     V_alpha      V_beta    J_alpha  \
0    CAVNAPTGTASKLTF   CASSMRSAVEQYF  TRAV8-1*01   TRBV19*01  TRAJ44*01   
1       CAVEGSQGNLIF   CASSMRSAVEQYF    TRAV2*01   TRBV19*01  TRAJ42*01   
2    CAGWPGSSNTGKLIF   CASSIRSLAEQYF   TRAV25*01   TRBV19*01  TRAJ37*01   
3  CAVRDAILTGGGNKLTF  CASRRQGITETQYF    TRAV3*01   TRBV27*01  TRAJ10*01   
4         CATEDNDMRF     CASSFSDTQYF   TRAV17*01  TRBV5-4*01  TRAJ43*01   

       J_beta    Epitope  clone_id          MHC dataset  Label  
0  TRBJ2-7*01  GILGFVFTL      25.0  HLA-A*02:01   viral      1  
1  TRBJ2-7*01  GILGFVFTL      27.0  HLA-A*02:01   viral      1  
2  TRBJ2-7*01  GILGFVFTL      34.0  HLA-A*02:01   viral      1  
3  TRBJ2-5*01  GILGFVFTL      35.0  HLA-A*02:01   viral      1  
4  TRBJ2-3*01  GILGFVFTL      39.0  HLA-A*02:01   viral      0  
{'d

TypeError: can't multiply sequence by non-int of type 'numpy.float64'